<a href="https://colab.research.google.com/github/Dhanushiya-tech/Portfolio-sample/blob/main/QAbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install \
gradio==4.44.0 \
ibm-watsonx-ai==1.1.2 \
langchain==0.2.11 \
langchain-community==0.2.10 \
langchain-ibm==0.1.11 \
chromadb==0.4.24 \
pypdf==4.3.1 \
pydantic==2.9.1 \
huggingface_hub==0.23.0

In [2]:
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
from ibm_watsonx_ai import Credentials

from langchain_ibm import WatsonxLLM, WatsonxEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA

import gradio as gr

# Optional: suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [3]:
from langchain_ibm import WatsonxLLM

def get_llm():
    model_id = "ibm/granite-3-2-8b-instruct"

    parameters = {
        "decoding_method": "sample",
        "temperature": 0.5,
        "max_new_tokens": 300,
        "top_k": 50,
        "top_p": 1
    }

    project_id = "skills-network"

    watsonx_llm = WatsonxLLM(
        model_id=model_id,
        url="https://us-south.ml.cloud.ibm.com",
        apikey="YOUR_API_KEY",   # 🔴 replace this
        project_id=project_id,
        params=parameters,
    )

    return watsonx_llm

In [4]:
from langchain_community.document_loaders import PyPDFLoader

def document_loader(file):
    loader = PyPDFLoader(file.name)
    loaded_document = loader.load()
    return loaded_document

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

In [6]:
from langchain_ibm import WatsonxEmbeddings

def watsonx_embedding():
    embed_params = {
        "truncate_input_tokens": 512
    }

    watsonx_embedding = WatsonxEmbeddings(
        model_id="ibm/slate-30m-english-rtrvr",
        url="https://us-south.ml.cloud.ibm.com",
        apikey="YOUR_API_KEY",        # 🔴 replace this
        project_id="skills-network",
        params=embed_params,
    )

    return watsonx_embedding

In [7]:
from langchain_community.vectorstores import Chroma

def vector_database(chunks):
    embedding_model = watsonx_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb

In [8]:
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    retriever = vectordb.as_retriever()
    return retriever

In [9]:
def retriever_qa(file, query):
    llm = get_llm()
    retriever_obj = retriever(file)

    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever_obj,
        return_source_documents=True
    )

    response = qa.invoke({"query": query})
    return response['result']


In [10]:
rag_application = gr.Interface(
    fn=retriever_qa,
    # allow_flagging="never",
    inputs=[
        gr.File(
            label="Upload PDF File",
            file_count="single",
            file_types=['.pdf'],
            type="filepath"
        ),
        gr.Textbox(
            label="Input Query",
            lines=2,
            placeholder="Type your question here..."
        )
    ],
    outputs=gr.Textbox(label="Answer"),
    title="📄 RAG PDF Question Answering Bot",
    description="Upload a PDF document and ask any question. The chatbot will try to answer using the provided document."
)

In [11]:
rag_application.launch(server_name="0.0.0.0", server_port=7860)

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://456406cef1df021dea.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [13]:
from langchain_ibm import WatsonxLLM, WatsonxEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# =========================
# LLM
# =========================
def get_llm():
    model_id = "ibm/granite-3-2-8b-instruct"

    parameters = {
        "decoding_method": "sample",
        "temperature": 0.5,
        "max_new_tokens": 300,
        "top_k": 50,
        "top_p": 1
    }

    project_id = "skills-network"

    watsonx_llm = WatsonxLLM(
        model_id=model_id,
        url="https://us-south.ml.cloud.ibm.com",
        apikey="YOUR_API_KEY",   # 🔴 replace this
        project_id=project_id,
        params=parameters,
    )
    return watsonx_llm


# =========================
# Document Loader
# =========================
def document_loader(file):
    loader = PyPDFLoader(file)
    loaded_document = loader.load()
    return loaded_document


# =========================
# Text Splitter
# =========================
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks


# =========================
# Embedding Model
# =========================
def watsonx_embedding():
    embed_params = {
        "truncate_input_tokens": 512
    }

    watsonx_embedding = WatsonxEmbeddings(
        model_id="ibm/slate-30m-english-rtrvr",
        url="https://us-south.ml.cloud.ibm.com",
        apikey="YOUR_API_KEY",   # 🔴 replace this
        project_id="skills-network",
        params=embed_params,
    )
    return watsonx_embedding


# =========================
# Vector DB
# =========================
def vector_database(chunks):
    embedding_model = watsonx_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb


# =========================
# Retriever
# =========================
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    retriever = vectordb.as_retriever()
    return retriever


# =========================
# QA Chain
# =========================
def retriever_qa(file, query):
    llm = get_llm()
    retriever_obj = retriever(file)

    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever_obj,
        return_source_documents=True
    )

    response = qa.invoke({"query": query})
    return response['result']


# =========================
# Gradio UI
# =========================
rag_application = gr.Interface(
    fn=retriever_qa,
    allow_flagging="never",
    inputs=[
        gr.File(label="Upload PDF File", file_count="single", file_types=['.pdf'], type="filepath"),
        gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")
    ],
    outputs=gr.Textbox(label="Answer"),
    title="📄 RAG PDF Question Answering Bot",
    description="Upload a PDF document and ask any question. The chatbot will try to answer using the provided document."
)

# =========================
# Launch App
# =========================
rag_application.launch()

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://ea8050d04462628c0c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [14]:
!pip install langchain langchain-community langchain-ibm chromadb ibm-watsonx-ai pypdf gradio